In [ ]:
from pathlib import Path
import re
from collections import Counter

import numpy as np
import pandas as pd


# ============================================================================
# 15. INSTITUTION SELF-MENTION LEAK AUDIT
# ============================================================================
# Deriving each institution's likely abbreviation from its name (the initials
# of its capitalized words) and searching the essays for that string flags
# 0 / 2619 essays, for two reasons: (1) the Institution
# field is formatted "Country - Institution Name" and the country prefix
# corrupted both the full-name match and the initials computation, and (2)
# real institutional abbreviations ("ULg" for Universite de Liege, "UCL" for
# Universite catholique de Louvain) are self-assigned brand short-forms, not
# reliably derivable from initials at all -- no naming heuristic reconstructs
# "ULg" from "Universite de Liege".
#
# This audit therefore uses no naming heuristic. Instead, for each institution (with
# enough essays to be worth testing) it finds short alphabetic tokens that
# are empirically disproportionate in THAT institution's essays compared to
# OTHER institutions of the SAME L1 (controlling for L1-wide topic effects,
# not just comparing across languages) -- the same frequency-ratio method
# 14's Check 4 used to originally surface "ulg"/"ucl"/"ucm", just grouped by
# institution instead of by language. This is how those terms were actually
# found, so it's a more principled fit for finding more like them than a
# name-derivation heuristic that already failed once.
#
# Still prints sample snippets for manual verification -- a frequency ratio
# is evidence, not proof, and small-institution counts make this noisy.
#
# Produces:
#   institution_leak_candidate_terms.csv (per institution, for audit)
#   institution_leak_flagged_essays.csv
#   institution_leak_summary_by_label.csv
#   institution_leak_accuracy_comparison.csv

BASE_DIR = Path("..").resolve()
RESULTS_DIR = BASE_DIR / "data/processed/results"
COMBINED_CSV_PATH = BASE_DIR / "data/processed/combined_nli_masked.csv"
METADATA_PATH = BASE_DIR / "data/raw/ICLEV3/metadata.csv"
LEAKAGE_FLAGGED_PATH = RESULTS_DIR / "leakage_flagged_essays.csv"
FULLTEXT_LLM_PATH = RESULTS_DIR / "llm_predictions_fulltext.csv"

OUTPUT_TERMS_PATH = RESULTS_DIR / "institution_leak_candidate_terms.csv"
OUTPUT_FLAGGED_PATH = RESULTS_DIR / "institution_leak_flagged_essays.csv"
OUTPUT_SUMMARY_PATH = RESULTS_DIR / "institution_leak_summary_by_label.csv"
OUTPUT_ACCURACY_PATH = RESULTS_DIR / "institution_leak_accuracy_comparison.csv"

ICLE_LANG_MAP = {
    "German": "DEU", "French": "FRA", "Italian": "ITA", "Japanese": "JPN",
    "Korean": "KOR", "Spanish": "SPA", "Turkish": "TUR", "Chinese": "ZHO",
}
LANGS = ["DEU", "FRA", "ITA", "JPN", "KOR", "SPA", "TUR", "ZHO"]

MIN_ESSAYS_PER_INSTITUTION = 8   # too few essays -> frequency ratios are meaningless noise
MIN_TOKEN_COUNT = 3              # token must appear at least this many times within the institution
MIN_RATIO = 5.0                  # at least this many times more common in-institution vs rest-of-L1
TOKEN_LEN_RANGE = (3, 10)        # institutional markers are usually short (abbreviations, city/college names)


def tokenize(s: str) -> list:
    return [t for t in re.findall(r"[a-z]+", s.lower()) if TOKEN_LEN_RANGE[0] <= len(t) <= TOKEN_LEN_RANGE[1]]


if not METADATA_PATH.exists() or not COMBINED_CSV_PATH.exists():
    print(f"Skipping -- need both {METADATA_PATH.name} and {COMBINED_CSV_PATH.name}.")
else:
    meta = pd.read_csv(METADATA_PATH)
    meta_target = meta[meta["Native language"].isin(ICLE_LANG_MAP)].copy()
    meta_target["label"] = meta_target["Native language"].map(ICLE_LANG_MAP)
    meta_target["doc_id"] = "ICLEV3_" + meta_target["File name"].astype(str).str.strip()
    if "Institution" not in meta_target.columns:
        print("No 'Institution' column in this metadata file -- nothing to check.")
    else:
        meta_target["Institution"] = meta_target["Institution"].fillna("").astype(str).str.strip()

        combined = pd.read_csv(COMBINED_CSV_PATH)
        combined["text"] = combined["text"].fillna("").astype(str)
        text_by_doc = dict(zip(combined["doc_id"], combined["text"]))
        meta_target["tokens"] = meta_target["doc_id"].map(lambda d: tokenize(text_by_doc.get(d, "")))

        term_rows = []
        flagged_records = []

        for l1_label, l1_group in meta_target.groupby("label"):
            # Word frequency across the WHOLE L1 sub-corpus, used as the "rest of L1" baseline
            # for each institution within it (per-institution vs per-institution comparisons on
            # small institutions are too noisy; comparing against the whole L1 pool is more stable).
            l1_total_counts = Counter()
            for toks in l1_group["tokens"]:
                l1_total_counts.update(set(toks))  # doc-frequency, not raw term-frequency, to avoid one long essay dominating
            n_docs_l1 = len(l1_group)

            for institution, inst_group in l1_group.groupby("Institution"):
                if len(inst_group) < MIN_ESSAYS_PER_INSTITUTION:
                    continue
                inst_doc_counts = Counter()
                for toks in inst_group["tokens"]:
                    inst_doc_counts.update(set(toks))
                n_docs_inst = len(inst_group)

                for token, inst_count in inst_doc_counts.items():
                    if inst_count < MIN_TOKEN_COUNT:
                        continue
                    rest_count = l1_total_counts[token] - inst_count
                    n_rest = n_docs_l1 - n_docs_inst
                    inst_rate = inst_count / n_docs_inst
                    rest_rate = (rest_count + 1) / max(n_rest, 1)  # +1 smoothing
                    ratio = inst_rate / rest_rate
                    if ratio >= MIN_RATIO:
                        term_rows.append(dict(
                            label=l1_label, institution=institution, token=token,
                            n_docs_institution=n_docs_inst, inst_count=inst_count, ratio=round(ratio, 1),
                        ))

        term_table = pd.DataFrame(term_rows).sort_values(["label", "institution", "ratio"], ascending=[True, True, False])
        term_table.to_csv(OUTPUT_TERMS_PATH, index=False)
        print(f"{len(term_table)} candidate institution-marker (label, institution, token) rows found "
              f"(ratio >= {MIN_RATIO}x, min {MIN_TOKEN_COUNT} occurrences, min {MIN_ESSAYS_PER_INSTITUTION} essays/institution).")
        print(f"Saved: {OUTPUT_TERMS_PATH}")
        if len(term_table):
            print("\nTop candidates per label (up to 5 shown):")
            for label, group in term_table.groupby("label"):
                top = group.sort_values("ratio", ascending=False).head(5)
                print(f"\n  {label}:")
                for _, r in top.iterrows():
                    print(f"    '{r['token']}' -> {r['institution']}  ({r['ratio']}x, n={r['inst_count']}/{r['n_docs_institution']} essays)")

        # Now flag essays: does the essay contain a term flagged as a marker for ITS OWN institution?
        marker_by_institution = {}
        for (label, institution), group in term_table.groupby(["label", "institution"]):
            marker_by_institution[(label, institution)] = set(group["token"])

        for _, row in meta_target.iterrows():
            key = (row["label"], row["Institution"])
            markers = marker_by_institution.get(key)
            if not markers:
                continue
            matched = markers.intersection(row["tokens"])
            if matched:
                flagged_records.append(dict(
                    doc_id=row["doc_id"], label=row["label"], institution=row["Institution"],
                    matched_terms=", ".join(sorted(matched)),
                ))

        flagged = pd.DataFrame(flagged_records)
        flagged.to_csv(OUTPUT_FLAGGED_PATH, index=False)
        print(f"\nFlagged {len(flagged)} / {len(meta_target)} ICLE essays containing a term that's empirically "
              "disproportionate to their OWN institution (within their own L1 sub-corpus).")
        print(f"Saved: {OUTPUT_FLAGGED_PATH}")

        if len(flagged):
            summary = flagged.groupby("label").size().reindex(LANGS, fill_value=0).rename("n_flagged").reset_index()
            totals = meta_target.groupby("label").size().reindex(LANGS, fill_value=0).rename("n_total")
            summary = summary.merge(totals, left_on="label", right_index=True)
            summary["pct_flagged"] = (summary["n_flagged"] / summary["n_total"] * 100).round(1)
            summary.to_csv(OUTPUT_SUMMARY_PATH, index=False)
            print("\n=== Institution self-mention rate by label ===")
            print(summary.to_string(index=False))
            print(f"Saved: {OUTPUT_SUMMARY_PATH}")

            print("\n=== Sample snippets for manual verification (up to 2 per label) ===")
            print("(Read these yourself -- a high frequency ratio is evidence, not proof, especially for small institutions.)")
            for label, group in flagged.groupby("label"):
                for _, r in group.head(2).iterrows():
                    text = text_by_doc.get(r["doc_id"], "")
                    term = r["matched_terms"].split(", ")[0]
                    m = re.search(r"\b" + re.escape(term) + r"\b", text, re.IGNORECASE)
                    if m:
                        start = max(0, m.start() - 60)
                        end = min(len(text), m.end() + 60)
                        snippet = text[start:end].replace("\n", " ")
                        print(f"\n[{label}] {r['doc_id']} ({r['institution']}) matched '{term}':\n  ...{snippet}...")

            if FULLTEXT_LLM_PATH.exists():
                llm_df = pd.read_csv(FULLTEXT_LLM_PATH)
                llm_df = llm_df[llm_df["error"].fillna("").eq("") & llm_df["predicted_l1"].notna()].copy()
                if "eval_partition" in llm_df.columns:
                    llm_df = llm_df[llm_df["eval_partition"] == "iclev3_external_test"].copy()

                self_ref_ids = set()
                if LEAKAGE_FLAGGED_PATH.exists():
                    prior = pd.read_csv(LEAKAGE_FLAGGED_PATH)
                    self_ref_ids = set(prior.loc[prior["self_referential_in_text"] == True, "doc_id"])
                llm_df = llm_df[~llm_df["doc_id"].isin(self_ref_ids)].copy()

                institution_leak_ids = set(flagged["doc_id"])
                llm_df["institution_self_mention"] = llm_df["doc_id"].isin(institution_leak_ids)
                llm_df["correct"] = llm_df["predicted_l1"] == llm_df["true_l1"]

                acc_split = llm_df.groupby("institution_self_mention")["correct"].agg(n="size", accuracy="mean").reset_index()
                acc_split.to_csv(OUTPUT_ACCURACY_PATH, index=False)
                print("\n=== LLM accuracy on the already self-disclosure-clean ICLE subset: institution self-mention vs. not ===")
                print(acc_split.to_string(index=False))
                print(f"Saved: {OUTPUT_ACCURACY_PATH}")

                triple_clean = llm_df[~llm_df["institution_self_mention"]]
                if len(triple_clean):
                    acc = triple_clean["correct"].mean()
                    print(
                        f"\nTriple-clean ICLE subset (no self-disclosure, no institution self-mention): "
                        f"n={len(triple_clean)}, accuracy={acc:.4f}"
                    )
                    print("If this is still ~90%+, institution leakage doesn't fully explain the residual either.")
            else:
                print(f"\nSkipping accuracy comparison -- {FULLTEXT_LLM_PATH.name} not found.")
        else:
            print(
                "\nNo essays flagged. This is a real possible outcome, not necessarily a bug -- but given 14's "
                "Check 4 found 'ulg'/'ucl'/'ucm' as disproportionate terms by hand, if this also comes back 0, "
                "inspect OUTPUT_TERMS_PATH: are candidate terms being generated at all, and is MIN_RATIO/"
                "MIN_TOKEN_COUNT too strict for your institution sizes?"
            )

print("\nDone.")
